# Step 3 (Kaggle): fair comparison — longer TFT training, 3 seeds, 365-day look-back, CAMELS-IND benchmark

Run with **Save Version → Save & Run All (Commit)**; about 5 hours. Settings: **GPU T4 x2**, **Internet On**.

| Model | What it tests |
|---|---|
| LSTM_ALL_S0/S1/S2 | Step 2's best model, 3 random seeds → how stable the result is (+ their ensemble) |
| LSTM365_ALL_S0/S1/S2 | same with a 365-day look-back (one full year of basin memory) |
| TFT_V2_LONG | TFT_V2 with about 2.3× more training data per epoch (1,200 batches of 256) |
| CAMELS_LSTM | LSTM predictions published with CAMELS-IND (benchmark from the dataset paper) |

Step 2 results are added as reference rows (`_STEP2`).

In [ ]:
import subprocess, sys, os

LOG = '/kaggle/working/results/run_log.txt'
os.makedirs('/kaggle/working/results', exist_ok=True)

def run(cmd):
    """Run a shell command, show its output live, save it to the log, and stop if it fails."""
    print(f'\n$ {cmd}', flush=True)
    with open(LOG, 'a') as log:
        log.write(f'\n$ {cmd}\n')
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
        p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'Step failed: {cmd} (see the output above or results/run_log.txt)')

run('nvidia-smi -L || echo no GPU')
run('cd /kaggle/working && (test -d flood || git clone -q https://github.com/yatharth7115/flood.git) && cd flood && git pull -q && git log --oneline -1')
os.chdir('/kaggle/working/flood')
run('pip install -q pytorch-forecasting pyyaml tabulate')

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/step2.yaml'))
cfg['data']['data_dir'] = '/tmp/camels_data'
cfg['data']['processed_file'] = '/tmp/camels_data/processed.parquet'
cfg['results_dir'] = '/kaggle/working/results'
os.makedirs('/tmp/camels_data', exist_ok=True)
yaml.safe_dump(cfg, open('configs/run.yaml', 'w'))
ALL = 'antecedent cn future_rain'

## 1. Data and the CAMELS-IND benchmark

In [ ]:
run('python src/download.py configs/run.yaml')
run('python src/dataset.py configs/run.yaml')
run('python src/benchmark_camels.py configs/run.yaml')

## 2. LSTM_ALL with 3 seeds (about 5 minutes each)

In [ ]:
for s in range(3):
    run(f'python src/train_lstm.py configs/run.yaml --name LSTM_ALL_S{s} --extra {ALL} --peak-weight 4 --seed {s}')

## 3. LSTM_ALL with a 365-day look-back, 3 seeds (about 15 minutes each)

In [ ]:
for s in range(3):
    run(f'python src/train_lstm.py configs/run.yaml --name LSTM365_ALL_S{s} --extra {ALL} --peak-weight 4 --seed {s} --lookback 365')

## 4. TFT_V2 with longer training (about 3.5 hours)

In [ ]:
run(f'python src/train_tft.py configs/run.yaml --name TFT_V2_LONG --extra {ALL} --peak-weight 4 --batch-size 256 --batches 1200 --epochs 15')

## 5. Compare

In [ ]:
run('python src/evaluate.py configs/run.yaml --reference results/step2/metrics_per_gauge.csv --ref-tag STEP2 --plot LSTM_ALL_ENS LSTM365_ALL_ENS TFT_V2_LONG CAMELS_LSTM')
from IPython.display import Image, display, Markdown
R = cfg['results_dir']
display(Markdown(open(f'{R}/summary.md').read()))
for f in ['nse_cdf.png', 'hydrograph_focus.png', 'tft_v2_long_variable_importance.png']:
    if os.path.exists(f'{R}/{f}'):
        display(Image(f'{R}/{f}', width=700))
run(f'cd {R} && rm -f *_model.pt')   # keep the output small